# Datamine ENVSEQ Process Example

This notebook demonstrates how to configure and run the **`envseq`** process wrapper in `dmstudio`.

## Process Description

## ENVSEQ

Process a block model generated from the [MODENV](<modenv.md>) process and evaluate the economics of mining satellite ore envelopes taking into account development, access and processing requirements to produce an extraction sequence.

The **MODENV** process outputs a grade (or value) model with additional fields * **ENVELOPE** and * **ENVNUM** to define the cell classification and each distinct spatial grouping of mined cells based on the optimization criteria:

1. maximum ore tonnes (equivalent to minimum included waste or dilution). If several minimum mining units (MMUs) have the same tonnes, then the MMU with the maximum grade is preferred.
2. maximum MMU head grade
3. maximum MMU contained metal
4. maximum accumulated value, where the supplied field is expressed in currency value eg dollar value.

In all cases except (4), the supplied field would be a grade (or equivalent grade for polymetallic deposits) that is handled as a density weighted average.

The **MODENV** process also outputs a &**RESULTS** file which summarizes the envelope statistics for tonnage, grade, value, extent and centre of gravity for each combination of envelope number * **ENVNUM** , cell classification * **ENVELOPE** and * **SHAPZONE**. Only the results for * **ENVELOPE** classification of **TOTAL** are used in the sequencing.

The mining sequence must take into account development, access and processing requirements. The criteria for ranking the envelope sequence can be selected from the same list of optimization criteria used to generate the MMU envelope, and a different criteria can be used to optimize the sequence.

The costs of sequential development of any satellite orebody from one that is already included in the extraction sequence are calculated from data in the costs penalty &**COSTS** file. The penalties are a function of the size and relative spatial position of the two MMU envelopes. The file must have a representative number of entries for the field size (**SIZE**), horizontal distance (**HORDIST**), vertical distance (**VERDIST**), and grade/value penalty to interpolate the initial fixed development cost (**PENALTY1**), and the variable haulage/extraction cost (**PENALTY2**).

The units of **PENALTY1** and **PENALTY2** depend on the optimizing criteria used in the the run of **ENVSEQ** :

1. ore tonnes
2. grade
3. metal content
4. accumulated value (eg dollar value)

For example if **OPTIMISE** =3 (metal content), then **PENALTY1** is the metal equivalent for the fixed cost development corresponding to **HORDIST** and **VERDIST** distances. **PENALTY2** is the metal equivalent per tonne per unit distance for the cost of transporting SIZE units of material over the **HORDIST** and **VERDIST** distances.

The envelope separation can be measured by centre of gravity (@**DISTMETH** =1), or closest distance between the envelope surfaces (@**DISTMETH** =2). Sufficient values must be included in the &**COSTS** file to allow the penalties to be interpolated for every combination of envelopes. Every combination of the selected values of **SIZE** , **HORDIST** and **VERDIST** must be included in the &**COSTS** file, and the file must be sorted in order of increasing **SIZE** , **HORDIST** and **VERDIST**. The penalty applied to an envelope to determine if it can be mined will take into account the **PENALTY1** to provide access from the prior envelope, and the cumulative haulage/extraction cost (sum of **PENALTY2** values in the sequence).

An envelope sequence file &**SEQUENCE** is used to predefine sequence relationships between envelopes that the user wants to predefine. Three types of relationships can be defined:

1. To specify one or more envelopes that will be developed at the start of the sequence (specify **ENVNUM1** and **SEQTYPE** =1). This option could be used to provide access through multiple shafts.
2. To specify that one envelope must be developed from another (specify **ENVNUM1** , **ENVNUM2** and **SEQTYPE** =2)
3. To specify that one envelope must be developed from another but allowing other intermediate envelopes (specify ENVNUM1, **ENVNUM2** and **SEQTYPE** =3)

If a &**SEQUENCE** file is not defined the process chooses the envelope that has the highest value based on the defined reference point, after penalties have been applied. If no reference point is defined then the envelope with the highest insitu value is chosen as the starting envelope.

The best mining sequence is reported in the &**OUT** file with pairs of envelope numbers **ENVNUM1** and **ENVNUM2**. The result can be visualized as an inverted tree structure (of parent-child relationships) with any envelope linked back to only one parent envelope. A parent envelope may have more than one child.

### Input Files:

* **envmodel** (Block Model):
  Input model file for evaluation. This will have been created as the output
  &**ENVMODEL** file by process **MODENV**. It must have the fields **XMORIG** ,
  **YMORIG** , **ZMORIG** , **NX** , **NY** , **NZ** (implicit), **IJK** , **XC** , **YC**
  and **ZC** (explicit). **XINC** , **YINC** and **ZINC** must exist as either explicit
  (sub-cells permitted) or implicit (no sub-cells). If it is a Rotated Model then it must
  also include the fields X0, Y0, Z0, ANGLE1, ANGLE2, ANGLE3, ROTAXIS1, ROTAXIS2, and
  ROTAXIS3. The file also has the fields ***GRADE, *VALUE, *ENVBEST, *DENSITY, *OPTIMISE,
  *ENVELOPE** and ***SHAPZONE** if specified. The value in the * **ENVBEST** field depend
  on the default value of the implicit field * **OPTIMISE** : 1 - (maximize ore tonnes)
  then the values in the two fields are ore tonnes. 2 - (maximize grade) then the values
  in the two fields are grade. 3 - (maximize metal) then the values in the two fields are
  metal content. 4 - (maximize dollars) then the values in the two fields are dollars.
  Required=Yes

* **results** (Undefined):
  Input results summary file to report statistics for each envelope, with fields

## ***ENVNUM, *ENVELOPE, *SHAPZONE, *GRADE, *VALUE, VOLUME, TONNES, MINX, MAXX, MINY, MAXY,

  MINZ, MAXZ, COGX, COGY, COGZ**. Each combination of * **ENVELOPE** (and * **SHAPZONE**
  if supplied) is included by **MODENV** but only those records where * **ENVELOPE** has
  the value '**TOTAL** ' are used in the sequencing.
  Required=Yes

* **costs** (Undefined):
  Input file to define the costs associated with alternate sequence combinations. The
  fields **SIZE** , **HORDIST** , **VERDIST** , **PENALTY1** , **PENALTY2** are required.
  The file must have all combinations of the discrete values selected for **SIZE** ,
  **HORDIST** and **VERDIST** and be sorted in the same field order.
  Required=Yes

* **sequence** (Undefined):
  Optional input file to define required sequence relationships between envelopes. Three
  fields **ENVNUM1** , **ENVNUM2** and **SEQTYPE** are required.
  Required=No

### Output Files:

* **out** (Undefined):
  Output sequence summary file, with pairs of sequence relationships in **ENVNUM1** and
  **ENVNUM2**. The initial envelope value **IVALUE** , the penalty values **PENALTY1** and
  **PENALTY2** to the previous envelope, and **PENALTY3** for prior envelopes and the
  final envelope value **FVALUE** are output. This file will also contain the centre of
  gravity of each envelope, if @**DISTMETH** =1, or the coordinates of the closest points,
  if @**DISTMETH** =2 stored in the fields (**X1,Y1,Z1**) and (**X2,Y2,Z2**).
  Required=Yes

### Fields:

* **grade** (Numeric : ENVMODEL):
  Numeric (explicit) field for the grade of input model blocks.
  Default=Undefined
  Required=No

* **value** (Numeric : ENVMODEL):
  Numeric (explicit) field for the value of input model blocks.
  Default=Undefined
  Required=No

* **shapzone** (Any : ENVMODEL):
  Field in the input model to distinguish zones.
  Default=Undefined
  Required=No

* **envbest** (Numeric : ENVMODEL):
  Numeric (explicit) field for the best envelope grade or value in the &ENVMODEL file.
  Default=ENVBEST
  Required=Yes

* **envelope** (Character : ENVMODEL):
  Alphanumeric (explicit) field for the cell envelope code in the &ENVMODEL file.
  Default=ENVELOPE
  Required=Yes

* **envnum** (Numeric : ENVMODEL):
  Numeric (explicit) field for the envelope number in the &ENVMODEL file.
  Default=ENVNUM
  Required=No

* **density** (Numeric : ENVMODEL):
  Optional density field in the input model for average grade and tonnage calculations.
  Default=Undefined
  Required=No

### Parameters:

* **hdgrade**:
  Required head grade for economic envelopes. The definition of head grade depends on the
  value of @**SEQOPT**. An absent value will cause the head grade test to be ignored.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **refz**:
  Reference elevation for calculation of penalties for the initial envelope. If a
  &**SEQUENCE** file is not defined then the reference elevation would normally be the
  surface elevation. The coordinate system for defining @**REFZ** is the unrotated system
  used in &**ENVMODEL**.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **refx**:
  Reference X coordinate for calculation of penalties for the initial envelope. If a
  &**SEQUENCE** file is not defined then the reference X coordinate would normally be the
  Easting of the existing or proposed shaft. The coordinate system for defining @**REFX**
  is the unrotated system used in &**ENVMODEL**. If the value is set to absent data (the
  default) then neither the X or Y reference coordinates are used.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **refy**:
  Reference Y coordinate for calculation of penalties for the initial envelope. If a
  &**SEQUENCE** file is not defined then the reference Y coordinate would normally be the
  Easting of the existing or proposed shaft. The coordinate system for defining @**REFY**
  is the unrotated system used in &**ENVMODEL**. If the value is set to absent data (the
  default) then neither the X or Y reference coordinates are used.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **seqopt**:
  Method to be used for ranking the envelope sequence: 0 - Use the same method used for
  optimizing the &**ENVMODEL** model. This is recorded as the default value of implicit
  field **OPTIMISE** in file &**ENVMODEL**. 1 - Maximize ore tonnes ie minimize [below
  cutoff] waste. 2 - Maximize grade 3 - Maximize contained metal 4 - Maximize accumulated
  value ie for dollar value.
  Range=0,4
  Values=0,1,2,3,4
  Default=0
  Required=No

* **distmeth**:
  Method for defining the envelope separation: 1 - measured by centre of gravity 2 -
  measured as the closest distance between the envelope surfaces.
  Range=1,2
  Values=1,2
  Default=1
  Required=Yes

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('envseq')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute envseq
print("Running envseq...")
dm_cmd.envseq(
    envmodel_i='t_mod1',  # required
    results_i=['t_input_file'],  # required
    costs_i=['t_input_file'],  # required
    out_o='t_envseq_out',  # required
    value_f='optional',  # required
    shapzone_f='optional',  # required
    envbest_f='optional',  # required
    envelope_f='optional',  # required
    density_f='optional',  # required
    hdgrade_p='required_val',  # required
    distmeth_p='required_val',  # required
    # sequence_i='optional',  # optional
    # grade_f='optional',  # optional
    # envnum_f='optional',  # optional
    # refz_p='optional',  # optional
    # refx_p='optional',  # optional
    # refy_p='optional',  # optional
    # seqopt_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("envseq execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_envseq_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")